# 07 DreaMS冻结谱图表示：批量提取与覆盖审计

本Notebook汇总冻结Development公共谱图的DreaMS批量embedding结果。它不读取ChemVista谱图，不进行外测，也不使用SMILES、Morgan、MACCS或分子图作为DreaMS输入。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs' / 'default.yaml').exists():
    raise RuntimeError(f'请从项目根目录执行Notebook，当前目录为：{ROOT}')
OUT = ROOT / 'outputs' / 'pbt_msms_v1' / 'dreams_v1' / 'batch_embeddings_v1'
ENV_MS = Path(r'D:\Appinstallation\Anaconda\envs\env_ms\python.exe')
ENV_DREAMS = Path(r'D:\Appinstallation\Anaconda\envs\env_dreams\python.exe')
print('project_root =', ROOT)
print('env_ms =', ENV_MS)
print('env_dreams =', ENV_DREAMS)
print('output_dir =', OUT)

## 可复现运行入口

批量重建默认关闭，避免查看结果时意外重复执行。需要从原始公共PKL重建时，将对应开关改为True。prepare必须使用env_ms；embed必须使用env_dreams；validate使用env_ms。

In [ ]:
RUN_PREPARE = False
RUN_EMBED = False
RUN_VALIDATE = False

run_env = dict(**__import__('os').environ)
run_env['PYTHONPATH'] = str(ROOT / 'src')
if RUN_PREPARE:
    subprocess.run([str(ENV_MS), '-m', 'pbt_msms.dreams_batch', '--stage', 'prepare'], cwd=ROOT, env=run_env, check=True)
if RUN_EMBED:
    subprocess.run([str(ENV_DREAMS), '-m', 'pbt_msms.dreams_batch', '--stage', 'embed', '--batch-size', '16'], cwd=ROOT, env=run_env, check=True)
if RUN_VALIDATE:
    subprocess.run([str(ENV_MS), '-m', 'pbt_msms.dreams_batch', '--stage', 'validate'], cwd=ROOT, env=run_env, check=True)
print({'RUN_PREPARE': RUN_PREPARE, 'RUN_EMBED': RUN_EMBED, 'RUN_VALIDATE': RUN_VALIDATE})

## 冻结运行摘要

In [ ]:
with (OUT / 'dreams_batch_run_manifest.json').open(encoding='utf-8') as handle:
    manifest = json.load(handle)
summary = pd.DataFrame([{
    'status': manifest['status'],
    'external_test_unlocked': manifest['external_test_unlocked'],
    'n_spectra': manifest['n_spectra'],
    'n_compounds': manifest['n_compounds'],
    'embedding_shape': str(manifest['embedding_shape']),
    'finite_fraction': manifest['embedding_stats']['finite_fraction'],
    'format_a_domain_spectra': manifest['n_format_a_domain_spectra'],
    'format_a_domain_compounds': manifest['n_format_a_domain_compounds'],
    'mvp_max_abs_diff': manifest['mvp_reproducibility']['max_abs_difference'],
    'gpu': manifest['embedding_environment']['gpu'],
    'elapsed_seconds': manifest['embedding_environment']['elapsed_seconds'],
}])
display(summary)

## 质量控制

In [ ]:
qc = pd.read_csv(OUT / 'dreams_batch_qc_checks.csv')
display(qc)
assert qc['status'].eq('PASS').all(), qc.loc[qc['status'].ne('PASS')]
print('QC:', qc['status'].value_counts().to_dict())

## 覆盖率与DreaMS官方DataFormatA适用域

全覆盖embedding与官方适用域子集必须分别报告。负离子、低信息谱图及m/z超域谱图不会被静默删除。

In [ ]:
coverage = pd.read_csv(OUT / 'dreams_batch_coverage_by_source_mode.csv')
display(coverage)
format_counts = pd.Series(manifest['format_a_failure_counts'], name='n_spectra').rename_axis('first_result').reset_index()
display(format_counts)

## embedding文件独立检查

In [ ]:
embeddings = np.load(OUT / 'development_dreams_embeddings.npy', mmap_mode='r', allow_pickle=False)
index = pd.read_csv(OUT / 'development_dreams_input_index.csv.gz')
print('embedding shape:', embeddings.shape)
print('embedding dtype:', embeddings.dtype)
print('index rows:', len(index))
print('compound count:', index['inchikey_2d'].nunique())
print('fold counts:', index['cv_fold'].value_counts().sort_index().to_dict())
assert embeddings.shape == (58687, 1024)
assert len(index) == len(embeddings)
assert np.array_equal(index['feature_row'].to_numpy(), np.arange(len(index)))

## 峰数差异审计

In [ ]:
discrepancies = pd.read_csv(OUT / 'dreams_batch_peak_count_discrepancies.csv')
print('n_discrepancies =', len(discrepancies))
if len(discrepancies):
    display(discrepancies[['spectrum_id', 'source_library', 'manifest_cleaned_peak_n', 'record_cleaned_peak_n', 'exported_peak_n']].head(20))
    assert (discrepancies['record_cleaned_peak_n'] == discrepancies['exported_peak_n']).all()

## 本步骤结论与下一步

DreaMS冻结表示已完成Development全量提取和无泄漏验收。下一步使用development_dreams_embeddings.npy与development_dreams_input_index.csv.gz，在既有Morgan–Butina结构簇五折上训练冻结embedding的浅层P/B/T预测头；ChemVista继续锁定。